# NLP Exercises (Part 2)

We have 2 exercises in this section. The exercises are:

4. Build your own Bag Of Words implementation using tokenizer created before.
5. Build a 5-gram model and clean up the results.

## Exercise 4. Build your own Bag Of Words implementation using tokenizer created before 

You need to implement following methods:

- ``fit_transform`` - gets a list of strings and returns matrix with it's BoW representation
- ``get_features_names`` - returns list of words corresponding to columns in BoW

In [ ]:
import numpy as np
import spacy

class BagOfWords:
    """Basic BoW implementation."""
    
    __nlp = spacy.load("en_core_web_sm")
    __bow_list = []
    
    def fit_transform(self, corpus: list):
        """Transform list of strings into BoW array.

        Parameters
        ----------
        corpus: List[str]
                Corpus of texts to be transformed

        Returns
        -------
        np.array
                Matrix representation of BoW

        """
        all_tokens = []
        for doc in corpus:
            tokens = [token.text.lower() for token in self.__nlp(doc) if token.is_alpha]
            all_tokens.extend(tokens)
        
        vocab = {}
        for token in all_tokens:
            if token not in vocab:
                vocab[token] = len(vocab)
        
        self.__bow_list = sorted(vocab.keys())
        
        bow_matrix = np.zeros((len(corpus), len(vocab)))
        for i, doc in enumerate(corpus):
            tokens = [token.text.lower() for token in self.__nlp(doc) if token.is_alpha]
            for token in tokens:
                if token in vocab:
                    bow_matrix[i, vocab[token]] += 1
        
        return bow_matrix
      

    def get_feature_names(self) -> list:
        """Return words corresponding to columns of matrix.

        Returns
        -------
        List[str]
                Words being transformed by fit function

        """   
        return self.__bow_list

corpus = [
     'Bag Of Words is based on counting',
     'words occurences throughout multiple documents.',
     'This is the third document.',
     'As you can see most of the words occur only once.',
     'This gives us a pretty sparse matrix, see below. Really, see below',
]    
    
vectorizer = BagOfWords()

X = vectorizer.fit_transform(corpus)
print(X)

print(vectorizer.get_feature_names())
print(len(vectorizer.get_feature_names()))

## Exercise 5. Build a 5-gram model and clean up the results.

There are three tasks to do:
1. Use 5-gram model instead of 3.
2. Change to capital letter each first letter of a sentence.
3. Remove the whitespace between the last word in a sentence and . ! or ?.

Hint: for 2. and 3. implement a function called ``clean_generated()`` that takes the generated text and fix both issues at once. It could be easier to fix the text after it's generated rather then doing some changes in the while loop.

In [ ]:
from nltk.book import *

wall_street = text7.tokens

import re
import random

N=5

tokens = wall_street

def cleanup():
    compiled_pattern = re.compile("^[a-zA-Z0-9.!?]")
    clean = list(filter(compiled_pattern.match,tokens))
    return clean
tokens = cleanup()

def build_ngrams():
    ngrams = []
    for i in range(len(tokens)-N+1):
        ngrams.append(tokens[i:i+N])
    return ngrams

def ngram_freqs(ngrams):
    counts = {}

    for ngram in ngrams:
        token_seq  = SEP.join(ngram[:-1])
        last_token = ngram[-1]

        if token_seq not in counts:
            counts[token_seq] = {}

        if last_token not in counts[token_seq]:
            counts[token_seq][last_token] = 0

        counts[token_seq][last_token] += 1;

    return counts

def next_word(text, N, counts):

    token_seq = SEP.join(text.split()[-(N-1):]);
    
    if token_seq not in counts:
        all_next_words = []
        for seq_dict in counts.values():
            all_next_words.extend(seq_dict.keys())
        if all_next_words:
            return random.choice(all_next_words)
        return "."
    
    choices = counts[token_seq].items();

    total = sum(weight for choice, weight in choices)
    r = random.uniform(0, total)
    upto = 0
    for choice, weight in choices:
        upto += weight;
        if upto > r: return choice
    assert False # should not reach here

In [ ]:
def clean_generated(text):
    text = re.sub(r'[/\\]', '', text)
    
    sentences = re.split(r'([.!?])', text)
    cleaned = []
    capitalize_next = True
    for part in sentences:
        if capitalize_next and part.strip():
            part = part[0].upper() + part[1:] if part else part
            capitalize_next = False
        if part in '.!?':
            capitalize_next = True
        cleaned.append(part)
    text = ''.join(cleaned)
    
    text = re.sub(r'\s+([.!?])', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

N=5

SEP=" "

sentence_count=5

ngrams = build_ngrams()
start_seq=None

counts = ngram_freqs(ngrams)

if start_seq is None: start_seq = random.choice(list(counts.keys()))
generated = start_seq.lower();

sentences = 0
while sentences < sentence_count:
    generated += SEP + next_word(generated, N, counts)
    sentences += 1 if generated.endswith(('.','!', '?')) else 0

cleaned_text = clean_generated(generated)
print(cleaned_text)